<a href="https://colab.research.google.com/github/IAmSipp/diabetes_prediction_with_NHANSES_dataset/blob/develop/NextDecade_Diabetes_Wearable_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
from google.colab import drive
drive.mount('/content/drive')
!ls "/content/drive/My Drive/TheNextDecade/Diabetes"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BPXO_L.xpt
diabetes_012_health_indicators_BRFSS2015.csv
diabetes_binary_5050split_health_indicators_BRFSS2015.csv
diabetes_binary_health_indicators_BRFSS2015.csv
TCHOL_L.xpt


In [36]:
!pip install pyreadstat requests

In [37]:
!pip install optuna

In [38]:
import pandas as pd
import numpy as np
import pyreadstat
import requests
from io import BytesIO
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import xgboost as xgb
import optuna
from imblearn.over_sampling import SMOTE
from sklearn.metrics import average_precision_score, roc_auc_score, classification_report

# **Data**

In [39]:
NHANSES_URLS = [
    'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DEMO_L.xpt',
    # 'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/BMX_L.xpt',
    'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/SLQ_L.xpt',
    'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/TCHOL_L.xpt',
    'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/PAQ_L.xpt',
    'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/BPXO_L.xpt',
    'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/GHB_L.xpt'
    ]

In [40]:
def download_nhanes_with_labels(url):
    response = requests.get(url)
    df, meta = pyreadstat.read_xport(BytesIO(response.content), encoding='latin1')
    return df, meta

def import_nhanes_xpt(urls):
    if isinstance(urls, str): urls = [urls]

    final_df = pd.DataFrame()
    all_labels = {}

    for i, url in enumerate(urls):
        df, meta = download_nhanes_with_labels(url)

        all_labels.update(meta.column_names_to_labels)

        if i == 0:
            final_df = df
        else:
            final_df = pd.merge(final_df, df, on='SEQN', how='inner')

    all_labels['SEQN'] = 'Patient_ID'
    final_df = final_df.rename(columns=all_labels)

    return final_df

In [41]:
raw_df = import_nhanes_xpt(NHANSES_URLS)
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6337 entries, 0 to 6336
Data columns (total 56 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Patient_ID                                6337 non-null   float64
 1   Data release cycle                        6337 non-null   float64
 2   Interview/Examination status              6337 non-null   float64
 3   Gender                                    6337 non-null   float64
 4   Age in years at screening                 6337 non-null   float64
 5   Age in months at screening - 0 to 24 mos  0 non-null      float64
 6   Race/Hispanic origin                      6337 non-null   float64
 7   Race/Hispanic origin w/ NH Asian          6337 non-null   float64
 8   Six-month time period                     6337 non-null   float64
 9   Age in months at exam - 0 to 19 years     264 non-null    float64
 10  Served active duty in US Armed Force

# **Data Reduction & Data Smapling**

In [42]:
selected_features = {
    # ข้อมูลพื้นฐาน
    'Patient_ID': 'ID',
    'Gender': 'Gender',
    'Age in years at screening': 'Age',

    # 'BMI Category - Children/Youth': 'BMI',

    # การนอน
    'Sleep hours - weekdays or workdays': 'Sleep_Hours',

    # กิจกรรม (นาทีต่อวัน)
    'Minutes sedentary activity': 'Sedentary_Minutes',
    'Minutes moderate LTPA': 'Moderate_Activity_Minutes',
    'Minutes vigorous LTPA': 'Vigorous_Activity_Minutes',

    # สุขภาพอื่นๆ (เอาไว้เสริมความแม่นยำ)
    'Total Cholesterol (mg/dL)': 'Cholesterol',

    # หัวใจและความดัน (เก็บมาทุกรอบก่อนเพื่อทำ Feature Engineering)
    'Pulse - 1st oscillometric reading': 'Pulse_1',
    'Pulse - 2nd oscillometric reading': 'Pulse_2',
    'Pulse - 3rd oscillometric reading': 'Pulse_3',
    'Systolic - 1st oscillometric reading': 'Sys_1',
    'Systolic - 2nd oscillometric reading': 'Sys_2',
    'Systolic - 3rd oscillometric reading': 'Sys_3',
    'Diastolic - 1st oscillometric reading': 'Dia_1',
    'Diastolic - 2nd oscillometric reading': 'Dia_2',
    'Diastolic - 3rd oscillometric reading': 'Dia_3',

    # ตัวแปรเป้าหมาย (Label)
    'Glycohemoglobin (%)': 'HbA1c'
}

In [43]:
selected_df = raw_df[selected_features.keys()].rename(columns=selected_features)

In [44]:
selected_df = selected_df.dropna(subset=['HbA1c'])

selected_df['Resting_HR'] = selected_df[['Pulse_2', 'Pulse_3']].mean(axis=1)
selected_df['Systolic_BP'] = selected_df[['Sys_2', 'Sys_3']].mean(axis=1)
selected_df['Diastolic_BP'] = selected_df[['Dia_2', 'Dia_3']].mean(axis=1)

selected_df['Is_Diabetes'] = (selected_df['HbA1c'] >= 6.5).astype(int)

cols_to_drop = ['Pulse_1', 'Pulse_2', 'Pulse_3', 'Sys_1', 'Sys_2', 'Sys_3', 'Dia_1', 'Dia_2', 'Dia_3', 'HbA1c']
selected_first_clean_df = selected_df.drop(columns=cols_to_drop)

In [45]:
selected_first_clean_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6002 entries, 0 to 6336
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   ID                         6002 non-null   float64
 1   Gender                     6002 non-null   float64
 2   Age                        6002 non-null   float64
 3   Sleep_Hours                5939 non-null   float64
 4   Sedentary_Minutes          5996 non-null   float64
 5   Moderate_Activity_Minutes  4781 non-null   float64
 6   Vigorous_Activity_Minutes  2725 non-null   float64
 7   Cholesterol                5714 non-null   float64
 8   Resting_HR                 5807 non-null   float64
 9   Systolic_BP                5807 non-null   float64
 10  Diastolic_BP               5807 non-null   float64
 11  Is_Diabetes                6002 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 609.6 KB


In [46]:
selected_first_clean_df

,ID,Gender,Age,Sleep_Hours,Sedentary_Minutes,Moderate_Activity_Minutes,Vigorous_Activity_Minutes,Cholesterol,Resting_HR,Systolic_BP,Diastolic_BP,Is_Diabetes
0,130378.0,1.0,43.0,9.5,360.0,45.0,45.0,264.0,80.5,131.5,95.0,0
1,130379.0,1.0,66.0,9.0,480.0,45.0,45.0,214.0,72.0,115.0,76.0,0
2,130380.0,2.0,44.0,8.0,240.0,20.0,NaN,187.0,80.0,108.0,78.0,0
3,130386.0,1.0,34.0,7.5,180.0,30.0,30.0,183.0,64.0,117.5,74.5,0
4,130387.0,2.0,68.0,3.0,1200.0,NaN,NaN,203.0,78.5,140.5,76.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
6331,142303.0,2.0,69.0,8.0,360.0,60.0,NaN,110.0,75.0,126.0,68.5,1
6332,142305.0,2.0,76.0,9.0,480.0,40.0,NaN,180.0,70.5,146.0,78.5,0
6333,142307.0,2.0,49.0,7.0,480.0,15.0,NaN,205.0,68.5,131.5,72.5,0
6335,142309.0,1.0,40.0,8.0,240.0,15.0,NaN,255.0,81.0,126.5,81.5,0


In [47]:
initial_count = len(selected_first_clean_df)

# กฎที่ 1: อายุ (Age) - NHANES ปกติจะดักที่ 80 ปีอยู่แล้ว แต่เรากันเหนียวไว้ที่ 100
condition_age = (selected_first_clean_df['Age'] >= 0) & (selected_first_clean_df['Age'] <= 100)

# กฎที่ 2: อัตราการเต้นหัวใจขณะพัก (Resting_HR)
# ตามสถิติ ต่ำกว่า 30 หรือสูงกว่า 200 ในขณะนั่งพักคือความผิดปกติของข้อมูลหรือภาวะฉุกเฉิน
condition_hr = (selected_first_clean_df['Resting_HR'] >= 30) & (selected_first_clean_df['Resting_HR'] <= 200)

# กฎที่ 3: ชั่วโมงการนอน (Sleep_Hours)
# การตอบว่านอน 0 ชั่วโมง หรือ 24 ชั่วโมง มักจะเป็นความผิดพลาดในการกรอกข้อมูล
condition_sleep = (selected_first_clean_df['Sleep_Hours'] >= 2) & (selected_first_clean_df['Sleep_Hours'] <= 16)

# กฎที่ 4: ความดันโลหิต (Blood Pressure)
# ค่าตัวบน (Systolic) ต้องมากกว่าตัวล่าง (Diastolic) และไม่อยู่ในระดับที่มนุษย์อยู่ไม่ได้
condition_bp = (selected_first_clean_df['Systolic_BP'] > selected_first_clean_df['Diastolic_BP']) & \
               (selected_first_clean_df['Systolic_BP'] <= 250) & \
               (selected_first_clean_df['Diastolic_BP'] >= 40)

# กฎที่ 5: กิจกรรม (Activity)
# นาทีที่นั่งนิ่งๆ หรือออกกำลังกาย ต้องไม่เกิน 1,440 นาที (24 ชม.)
condition_activity = (selected_first_clean_df['Sedentary_Minutes'] <= 1440) & \
                     (selected_first_clean_df['Moderate_Activity_Minutes'] <= 1440)

# 2. รวมกฎทั้งหมดเข้าด้วยกัน
# ใช้เครื่องหมาย & (AND) เพื่อบอกว่าต้องผ่านทุกกฎ
all_conditions = condition_age & condition_hr & condition_sleep & condition_bp & condition_activity

# 3. กรองข้อมูล (Filter)
# เราจะเลือกเฉพาะแถวที่ผ่านทุกเงื่อนไข หรือเป็นค่าว่าง (NaN)
# *หมายเหตุ: เราจะไม่ลบ NaN ที่นี่ เพราะเดี๋ยวเราจะใช้ Imputer จัดการใน Pipeline*
remove_outliner_df = selected_first_clean_df[all_conditions | selected_first_clean_df.isna().any(axis=1)]

# 4. ตรวจสอบผล
final_count = len(remove_outliner_df)
print(f"ลบข้อมูลที่ผิดปกติออกไป: {initial_count - final_count} แถว")
print(f"คงเหลือข้อมูลคุณภาพ: {final_count} แถว")

ลบข้อมูลที่ผิดปกติออกไป: 8 แถว
คงเหลือข้อมูลคุณภาพ: 5994 แถว


In [48]:
remove_outliner_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5994 entries, 0 to 6336
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   ID                         5994 non-null   float64
 1   Gender                     5994 non-null   float64
 2   Age                        5994 non-null   float64
 3   Sleep_Hours                5931 non-null   float64
 4   Sedentary_Minutes          5988 non-null   float64
 5   Moderate_Activity_Minutes  4773 non-null   float64
 6   Vigorous_Activity_Minutes  2717 non-null   float64
 7   Cholesterol                5706 non-null   float64
 8   Resting_HR                 5799 non-null   float64
 9   Systolic_BP                5799 non-null   float64
 10  Diastolic_BP               5799 non-null   float64
 11  Is_Diabetes                5994 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 608.8 KB


In [49]:
X_not_cleaned = remove_outliner_df.drop(columns=['Is_Diabetes'])
y_not_cleand = remove_outliner_df['Is_Diabetes']

X_train_not_clean, X_test_not_clean, y_train_not_clean, y_test_not_clean = \
  train_test_split(X_not_cleaned, y_not_cleand, test_size=0.2, random_state=42, stratify=y_not_cleand)

In [50]:
train_not_clean_df = pd.concat([X_train_not_clean, y_train_not_clean], axis=1)
test_not_clean_df = pd.concat([X_test_not_clean, y_test_not_clean], axis=1)

In [51]:
train_not_clean_df.to_csv("diabetes_train_not_clean_data.csv", index=False)
test_not_clean_df.to_csv("diabetes_test_not_clean_data.csv", index=False)

In [52]:
def drop_statistical_outliers(df, columns, threshold=3):
    """
    ลบแถวที่เป็น Outliers โดยใช้ Z-Score
    threshold=3 หมายถึง ข้อมูลที่อยู่ห่างจากค่าเฉลี่ยเกิน 3 เท่าของส่วนเบี่ยงเบนมาตรฐาน
    """
    df_cleaned = df.copy()
    for col in columns:
        if col in df_cleaned.columns:
            mean = df_cleaned[col].mean()
            std = df_cleaned[col].std()

            # สร้างขอบเขตบนและล่าง
            lower_bound = mean - (threshold * std)
            upper_bound = mean + (threshold * std)

            # กรองข้อมูล
            df_cleaned = df_cleaned[(df_cleaned[col] >= lower_bound) & (df_cleaned[col] <= upper_bound)]

    print(f"ลบ Outliers ทางสถิติออกไปทั้งหมด: {len(df) - len(df_cleaned)} แถว")
    return df_cleaned

In [53]:
def create_preprocessing_pipeline():
    """
    สร้าง Pipeline สำหรับจัดการข้อมูลตัวเลข
    - Imputation: เติมค่าว่างด้วย Median (มัธยฐาน) เพราะทนต่อ Outliers ได้ดีกว่า Mean
    - Scaling: ปรับข้อมูลให้เป็น Standard Scale (Mean=0, Std=1)
    """
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    return numeric_transformer

In [54]:
# 1. โหลดข้อมูลที่คุณ Save ไว้ (หรือจากตัวแปรเดิม)
# train_df = pd.read_csv("diabetes_train_data.csv")

# 2. ทำ Statistical Drop (ทำเฉพาะในชุด Train เท่านั้น!)
# เลือกเฉพาะคอลัมน์ที่เป็นตัวเลขและมีโอกาสเกิด Outliers สูง
cols_to_check = ['Resting_HR', 'Sedentary_Minutes', 'Age', 'Sleep_Hours']
train_df_cleaned = drop_statistical_outliers(train_not_clean_df, cols_to_check, threshold=3)

# 3. แยก Features และ Label
X_train = train_df_cleaned.drop(columns=['ID', 'Is_Diabetes'])
y_train = train_df_cleaned['Is_Diabetes']

X_test = test_not_clean_df.drop(columns=['ID', 'Is_Diabetes'])
y_test = test_not_clean_df['Is_Diabetes']

# 4. สร้างและรัน Pipeline
preprocessor = create_preprocessing_pipeline()

# เรียนรู้จากชุด Train และแปลงข้อมูล
X_train_final = preprocessor.fit_transform(X_train)

# ใช้ค่าความรู้จากชุด Train ไปแปลงชุด Test (ห้าม fit ชุด Test!)
X_test_final = preprocessor.transform(X_test)

# แปลงผลลัพธ์กลับเป็น DataFrame (เพื่อให้ดูชื่อคอลัมน์ได้ง่ายขึ้น)
X_train_final = pd.DataFrame(X_train_final, columns=X_train.columns)
X_test_final = pd.DataFrame(X_test_final, columns=X_test.columns)

print("ข้อมูลพร้อมสำหรับการเทรนแล้ว!")
X_train_final.head()

ลบ Outliers ทางสถิติออกไปทั้งหมด: 315 แถว
ข้อมูลพร้อมสำหรับการเทรนแล้ว!


,Gender,Age,Sleep_Hours,Sedentary_Minutes,Moderate_Activity_Minutes,Vigorous_Activity_Minutes,Cholesterol,Resting_HR,Systolic_BP,Diastolic_BP
0,0.917663,-0.363557,0.192559,0.567978,-0.035902,-0.158383,1.045653,-0.009100,-0.651413,-0.007537
1,0.917663,0.246207,0.192559,0.567978,-0.035902,-0.105743,-0.610666,-0.561138,0.241982,0.817097
2,-1.089725,-1.195053,-1.922678,1.734893,-0.035902,-0.053103,-0.706684,1.604552,-0.930599,-0.053350
3,0.917663,-0.585289,-0.865060,0.859707,-0.080512,-0.189968,1.333709,1.010049,0.074470,1.137788
4,-1.089725,-1.860250,-0.512520,-0.015480,-0.035902,-0.000463,0.133477,1.094978,0.046552,-0.648919


In [55]:
smote = SMOTE(random_state=42)
X_resampled_train, y_resampled_train = smote.fit_resample(X_train_final, y_train)

In [56]:
X_resampled_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7908 entries, 0 to 7907
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Gender                     7908 non-null   float64
 1   Age                        7908 non-null   float64
 2   Sleep_Hours                7908 non-null   float64
 3   Sedentary_Minutes          7908 non-null   float64
 4   Moderate_Activity_Minutes  7908 non-null   float64
 5   Vigorous_Activity_Minutes  7908 non-null   float64
 6   Cholesterol                7908 non-null   float64
 7   Resting_HR                 7908 non-null   float64
 8   Systolic_BP                7908 non-null   float64
 9   Diastolic_BP               7908 non-null   float64
dtypes: float64(10)
memory usage: 617.9 KB


In [57]:
y_resampled_train.info()

<class 'pandas.core.series.Series'>
RangeIndex: 7908 entries, 0 to 7907
Series name: Is_Diabetes
Non-Null Count  Dtype
--------------  -----
7908 non-null   int64
dtypes: int64(1)
memory usage: 61.9 KB


# **Find Best Hyperparameters**

In [58]:
# ปิดข้อความแจ้งเตือนยิบย่อยของ optuna เพื่อให้ผลลัพธ์บนหน้าจอสะอาดตา
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [59]:
def objective(trial):
    # 1. กำหนดขอบเขตขีดความสามารถที่ต้องการให้ Optuna ลองสุ่มจูน (Fine-tune)
    params = {
        'verbosity': 0,
        'objective': 'binary:logistic',
        'eval_metric': 'aucpr', # โฟกัสพื้นที่ใต้กราฟ Precision-Recall
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'alpha': trial.suggest_float('alpha', 1e-3, 10.0, log=True),
        'lambda': trial.suggest_float('lambda', 1e-3, 10.0, log=True),
        'scale_pos_weight': 1 # ตั้งเป็น 1 เสมอเพราะฝั่ง Train เราถูกปรับสมดุลด้วย SMOTE แล้ว
    }

    # 2. ปรับสมดุลข้อมูล Train เฉพาะในลูปทดสอบนี้ด้วย SMOTE (1:1)
    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X_train_final, y_train)

    # 3. สร้างโมเดลและเทรนด้วยข้อมูลที่สมดุลแล้ว
    model = xgb.XGBClassifier(**params)
    model.fit(X_resampled, y_resampled)

    # 4. นำไปทำนายบน Test Set จริง (ข้อมูลดิบดั้งเดิม ไม่ผ่าน SMOTE)
    y_proba = model.predict_proba(X_test_final)[:, 1]

    # 5. คำนวณคะแนนส่งกลับไปให้ Optuna ประเมิน (ยิ่งเข้าใกล้ 1.0 ยิ่งดี)
    score = average_precision_score(y_test, y_proba)
    return score

In [60]:
print("กำลังเริ่มต้นสแกนหารูปแบบโมเดลที่ดีที่สุด (150 Trials)...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=150)

print("\nค้นหาพารามิเตอร์เสร็จสิ้น!")
print("Best PR-AUC Score ที่ทำได้:", round(study.best_value, 4))
print("ค่าพารามิเตอร์ที่ดีที่สุดสำหรับข้อมูลของคุณ:")
for key, value in study.best_params.items():
    print(f"   - {key}: {value}")

กำลังเริ่มต้นสแกนหารูปแบบโมเดลที่ดีที่สุด (150 Trials)...

ค้นหาพารามิเตอร์เสร็จสิ้น!
Best PR-AUC Score ที่ทำได้: 0.3554
ค่าพารามิเตอร์ที่ดีที่สุดสำหรับข้อมูลของคุณ:
   - learning_rate: 0.002371742062209093
   - max_depth: 3
   - n_estimators: 579
   - subsample: 0.5566615523360138
   - colsample_bytree: 0.9061708995341176
   - min_child_weight: 10
   - alpha: 0.3256431137078469
   - lambda: 0.0544638360693873


# **Train**

In [61]:
print("\n เริ่มต้นเทรนโมเดลตัวจริงด้วยค่าพารามิเตอร์ที่ดีที่สุด...")

# 1. ทำ SMOTE บนชุด Train หลักเพื่อให้โมเดลสุดท้ายเรียนรู้เต็มที่
smote_final = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote_final.fit_resample(X_train_final, y_train)


 เริ่มต้นเทรนโมเดลตัวจริงด้วยค่าพารามิเตอร์ที่ดีที่สุด...


In [62]:
final_params = study.best_params
final_params['objective'] = 'binary:logistic'
final_params['eval_metric'] = 'aucpr'
final_params['scale_pos_weight'] = 0.73

In [63]:
final_model = xgb.XGBClassifier(**final_params)
final_model.fit(X_train_resampled, y_train_resampled)

print("โมเดลสุดท้ายเสร็จสมบูรณ์และพร้อมทำนายผลแล้ว!")

โมเดลสุดท้ายเสร็จสมบูรณ์และพร้อมทำนายผลแล้ว!


# **Test**

In [64]:
# ===================================================
# ทดสอบแปลงผลลัพธ์เป็น "คะแนนความเสี่ยงเบาหวาน (0-100)"
# ===================================================
# ดึงค่าความน่าจะเป็นคอลัมน์ที่ 1 (โอกาสที่เป็นเบาหวาน)
y_test_proba = final_model.predict_proba(X_test_final)[:, 1]
risk_scores = y_test_proba * 100

In [65]:
# จัดหน้าตารางสรุปผลมาลองเช็คดูตัวเลข
evaluation_df = pd.DataFrame({
    'Actual_Diabetes': y_test.values,
    'Risk_Score_0_100': np.round(risk_scores, 2)
})

print("\n📊 ตัวอย่างคะแนนความเสี่ยงที่โมเดลคำนวณออกมาได้ (10 คนแรกใน Test Set):")
print(evaluation_df.head(10).to_string(index=False))


📊 ตัวอย่างคะแนนความเสี่ยงที่โมเดลคำนวณออกมาได้ (10 คนแรกใน Test Set):
 Actual_Diabetes  Risk_Score_0_100
               0         41.439999
               0         14.960000
               0         26.490000
               0         27.809999
               1         63.139999
               0         14.130000
               0         25.770000
               0         17.660000
               0         48.570000
               0         19.629999


In [66]:
# ดูภาพรวมความแม่นยำเพิ่มเติมแบบ Hard Predict
y_pred_hard = (y_test_proba >= 0.5).astype(int)
print("\n📝 รายงานประสิทธิภาพภาพรวม (Classification Report):")
print(classification_report(y_test, y_pred_hard))


📝 รายงานประสิทธิภาพภาพรวม (Classification Report):
              precision    recall  f1-score   support

           0       0.93      0.82      0.87      1055
           1       0.30      0.56      0.39       144

    accuracy                           0.79      1199
   macro avg       0.62      0.69      0.63      1199
weighted avg       0.86      0.79      0.82      1199

